# DAC Post-Training (Standalone, Audio-Folder Input)

This notebook runs a lightweight DAC post-training experiment (post-filter) on Hindi Audio on Indic Voices Dataset.

Official references used:
- DAC: https://github.com/descriptinc/descript-audio-codec
- IndicConformer 600M: https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual
- IndicVoices: https://huggingface.co/datasets/ai4bharat/IndicVoices

In [ ]:
# Install required libraries
!pip -q install -U pip setuptools wheel
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install descript-audio-codec==1.0.0 transformers datasets accelerate huggingface_hub
!pip -q install numpy pandas scipy librosa soundfile tqdm jiwer pystoi pesq

In [ ]:
!pip install onnxruntime

In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
import soundfile as sf
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', DEVICE)

Device: cuda


In [ ]:
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

Mounted at /content/drive
Drive mounted at /content/drive


In [ ]:
# ==== USER SETTINGS ====
# Auto-detect dataset root across common local/Colab locations.
CANDIDATE_DATA_ROOTS = [
    Path('/content/data'),                           # Colab upload to runtime
    Path('/content/drive/MyDrive/data'),             # Colab Drive mount
    Path('/data'),                                   # symlinked/system path
    Path.cwd() / 'data',                             # local workspace
    Path('/content') / 'fine-tuning' / 'data',       # repo copied under /content
 ]

DATA_ROOT = next((p for p in CANDIDATE_DATA_ROOTS if p.exists()), CANDIDATE_DATA_ROOTS[0])
AUDIO_ROOT = DATA_ROOT / 'audios' / 'hindi'
TRANSCRIPT_CSV = DATA_ROOT / 'transcriptions.csv'  # optional

SR = 16000
SEGMENT_SECONDS = 2.0
TRAIN_HOURS = 2
VAL_MIN = 20
TEST_MIN = 20

BATCH_SIZE = 2
EPOCHS = 4
LR = 1e-4
GRAD_ACCUM = 4

MAX_EVAL_FILES = 40
RUN_ASR = True  # set True only if transcript + model access are available

OUT_DIR = Path('/content/nb_outputs') if Path('/content').exists() else (Path.cwd() / 'outputs' / 'nb_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Data root:', DATA_ROOT, '| exists:', DATA_ROOT.exists())
print('Audio root:', AUDIO_ROOT, '| exists:', AUDIO_ROOT.exists())
print('Transcript CSV:', TRANSCRIPT_CSV, '| exists:', TRANSCRIPT_CSV.exists())
print('Output dir:', OUT_DIR)

if not AUDIO_ROOT.exists():
    print('WARNING: AUDIO_ROOT not found. If on Colab, ensure your uploaded folder is /content/data/audios/hindi')

Data root: /content/drive/MyDrive/data | exists: True
Audio root: /content/drive/MyDrive/data/audios/hindi | exists: True
Transcript CSV: /content/drive/MyDrive/data/transcriptions.csv | exists: True
Output dir: /content/nb_outputs


In [ ]:
AUDIO_EXTS = {'.wav', '.flac', '.mp3', '.m4a', '.ogg'}

def load_audio_mono(path, target_sr=16000):
    wav, sr = sf.read(str(path), dtype='float32', always_2d=False)
    if wav.ndim > 1:
        wav = np.mean(wav, axis=1)
    if int(sr) != int(target_sr):
        wav = librosa.resample(wav, orig_sr=int(sr), target_sr=int(target_sr))
    wav = np.clip(wav, -1.0, 1.0).astype(np.float32)
    return wav

def duration_seconds(path):
    info = sf.info(str(path))
    return float(info.frames) / float(max(1, info.samplerate))

def slice_or_pad(wav, sr=16000, seconds=2.0):
    n = int(sr * seconds)
    if len(wav) < n:
        out = np.zeros(n, dtype=np.float32)
        out[:len(wav)] = wav
        return out
    if len(wav) == n:
        return wav
    s = random.randint(0, len(wav) - n)
    return wav[s:s+n]

def norm_rel(p):
    s = str(p).replace('\\', '/').lstrip('./')
    while s.startswith('/'):
        s = s[1:]
    return s

transcript_map = {}
if TRANSCRIPT_CSV.exists():
    df_t = pd.read_csv(TRANSCRIPT_CSV)
    if {'audio_path', 'transcript'}.issubset(df_t.columns):
        for _, r in df_t.iterrows():
            k = norm_rel(r['audio_path'])
            t = str(r['transcript'])
            transcript_map[k] = t
            # Also index by filename-only fallback for minor path format mismatches.
            transcript_map.setdefault(Path(k).name, t)

audio_files = sorted([p for p in AUDIO_ROOT.rglob('*') if p.suffix.lower() in AUDIO_EXTS])
print('Found files:', len(audio_files))

rows = []
matched_transcripts = 0
for p in audio_files:
    rel_from_audio_root = norm_rel(p.relative_to(AUDIO_ROOT))
    rel_from_data_root = norm_rel(p.relative_to(DATA_ROOT)) if DATA_ROOT in p.parents else rel_from_audio_root
    rel_from_audios = norm_rel(p.relative_to(DATA_ROOT / 'audios')) if (DATA_ROOT / 'audios') in p.parents else rel_from_audio_root

    transcript = (
        transcript_map.get(rel_from_audio_root)
        or transcript_map.get(rel_from_data_root)
        or transcript_map.get(rel_from_audios)
        or transcript_map.get(p.name, '')
    )
    if transcript:
        matched_transcripts += 1

    rows.append({
        'path': p,
        'rel': rel_from_audio_root,
        'transcript': transcript,
        'dur': duration_seconds(p)
    })

print('Transcript matches:', matched_transcripts, '/', len(rows))

random.shuffle(rows)
train_target = TRAIN_HOURS * 3600
val_target = VAL_MIN * 60
test_target = TEST_MIN * 60

train, val, test = [], [], []
td = vd = ed = 0.0
for r in rows:
    if td < train_target:
        train.append(r); td += r['dur']
    elif vd < val_target:
        val.append(r); vd += r['dur']
    elif ed < test_target:
        test.append(r); ed += r['dur']

print('Split sizes:', len(train), len(val), len(test))
print('Durations -> train_h:', round(td/3600,3), 'val_m:', round(vd/60,2), 'test_m:', round(ed/60,2))

Found files: 4740
Transcript matches: 4740 / 4740
Split sizes: 1300 206 212
Durations -> train_h: 2.002 val_m: 20.11 test_m: 20.0


In [ ]:
def split_into_chunks(wav, sr=16000, seconds=2.0):
    n = int(sr * seconds)
    chunks = []
    for i in range(0, len(wav), n):
        chunk = wav[i:i+n]
        if len(chunk) < n:
            break
        chunks.append(chunk)
    return chunks

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, items, sr=16000, seconds=2.0):
        self.sr = sr
        self.seconds = seconds
        self.data = []

        print("🔄 Preprocessing into chunks...")

        for it in items:
            wav = load_audio_mono(it['path'], self.sr)
            chunks = split_into_chunks(wav, self.sr, self.seconds)

            for ch in chunks:
                self.data.append({
                    'wav': ch,
                    'transcript': it['transcript'],
                    'path': str(it['path'])
                })

        print(f"✅ Total chunks created: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        wav = self.data[idx]['wav'] # Move wav assignment here
        if random.random() < 0.3:
          noise = np.random.normal(0, 0.003, size=wav.shape).astype(np.float32)
          wav = wav + noise
        x = torch.from_numpy(wav).unsqueeze(0)

        return {
            'clean': x,
            'transcript': self.data[idx]['transcript'],
            'path': self.data[idx]['path']
        }
def collate_fn(batch):
    clean = torch.stack([b['clean'] for b in batch], dim=0)
    return {
        'clean': clean,
        'transcript': [b['transcript'] for b in batch],
        'path': [b['path'] for b in batch],
    }

train_loader = DataLoader(AudioDataset(train, SR, SEGMENT_SECONDS), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(AudioDataset(val, SR, SEGMENT_SECONDS), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
test_loader  = DataLoader(AudioDataset(test, SR, SEGMENT_SECONDS), batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)
print('Loaders ready')

🔄 Preprocessing into chunks...
✅ Total chunks created: 2968
🔄 Preprocessing into chunks...
✅ Total chunks created: 501
🔄 Preprocessing into chunks...
✅ Total chunks created: 502
Loaders ready


In [ ]:
import dac

# Official DAC weight download/load
model_path = dac.utils.download(model_type='16khz')
dac_model = dac.DAC.load(model_path).to(DEVICE).eval()
for p in dac_model.parameters():
    p.requires_grad = False

class ResidualBlock(nn.Module):
    def __init__(self, channels=128, kernel_size=7, dilation=1):
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation),
        )

    def forward(self, x):
        return x + self.net(x)

class PostFilter(nn.Module):
    def __init__(self, channels=192, blocks=10, kernel_size=7):
        super().__init__()
        self.inp = nn.Conv1d(1, channels, 1)
        body = []
        for i in range(blocks):
            body += [ResidualBlock(channels, kernel_size, 2 ** (i % 4)), nn.LeakyReLU(0.2, inplace=True)]
        self.body = nn.Sequential(*body)
        self.out = nn.Conv1d(channels, 1, 1)

    def forward(self, x):
        y = self.out(self.body(self.inp(x)))
        return torch.clamp(x + 0.5*y, -1.0, 1.0)

postfilter = PostFilter().to(DEVICE)
print('DAC + postfilter ready')

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


DAC + postfilter ready


In [ ]:
def align_len(a, b):
    m = min(a.shape[-1], b.shape[-1])
    return a[..., :m], b[..., :m]

class MRSTFTLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.res = [(1024, 256, 1024), (512, 128, 512), (256, 64, 256)]

    def forward(self, pred, target):
        eps = 1e-7
        total = 0.0
        for n_fft, hop, win in self.res:
            w = torch.hann_window(win, device=pred.device)
            p = torch.stft(pred.squeeze(1), n_fft=n_fft, hop_length=hop, win_length=win, window=w, return_complex=True)
            t = torch.stft(target.squeeze(1), n_fft=n_fft, hop_length=hop, win_length=win, window=w, return_complex=True)
            pm = torch.sqrt(p.real**2 + p.imag**2 + eps)
            tm = torch.sqrt(t.real**2 + t.imag**2 + eps)
            total += torch.mean(torch.abs(torch.log(pm + eps) - torch.log(tm + eps)))
        return total / len(self.res)

l1 = nn.L1Loss()
mr = MRSTFTLoss()
opt = torch.optim.AdamW(postfilter.parameters(), lr=LR, betas=(0.9, 0.95))

best_val = 1e9
for ep in range(EPOCHS):
    postfilter.train()
    opt.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f'train epoch {ep+1}/{EPOCHS}')
    for i, batch in enumerate(pbar):
        clean = batch['clean'].to(DEVICE)
        with torch.no_grad():
            base = dac_model(clean, sample_rate=SR)['audio']
        base, clean = align_len(base, clean)
        enh = postfilter(base)
        loss = (0.6 * l1(enh, clean) + 0.4 * mr(enh, clean)) / GRAD_ACCUM
        loss.backward()

        if (i + 1) % GRAD_ACCUM == 0:
            opt.step()
            opt.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=float(loss.item() * GRAD_ACCUM))

    postfilter.eval()
    vals = []
    with torch.no_grad():
        for batch in val_loader:
            clean = batch['clean'].to(DEVICE)
            base = dac_model(clean, sample_rate=SR)['audio']
            base, clean = align_len(base, clean)
            enh = postfilter(base)
            vals.append(float((l1(enh, clean) + mr(enh, clean)).item()))

    vloss = float(np.mean(vals)) if vals else 0.0
    print('val_loss:', vloss)

    torch.save({'model': postfilter.state_dict(), 'val_loss': vloss}, OUT_DIR / 'postfilter_last.pt')
    if vloss < best_val:
        best_val = vloss
        torch.save({'model': postfilter.state_dict(), 'val_loss': vloss}, OUT_DIR / 'postfilter_best.pt')

print('Training done. Best val:', best_val)

train epoch 1/4: 100%|██████████| 1484/1484 [01:12<00:00, 20.50it/s, loss=0.331]


val_loss: 0.7009288805177012


train epoch 2/4: 100%|██████████| 1484/1484 [01:10<00:00, 21.12it/s, loss=0.313]


val_loss: 0.6813521052736685


train epoch 3/4: 100%|██████████| 1484/1484 [01:10<00:00, 21.17it/s, loss=0.346]


val_loss: 0.6823570385159724


train epoch 4/4: 100%|██████████| 1484/1484 [01:10<00:00, 21.14it/s, loss=0.217]


val_loss: 0.6760591638990607
Training done. Best val: 0.6760591638990607


In [17]:
from pesq import pesq
from pystoi import stoi

def si_sdr(ref, est, eps=1e-8):
    alpha = np.dot(est, ref) / (np.dot(ref, ref) + eps)
    t = alpha * ref
    n = est - t
    return 10 * np.log10((np.sum(t*t) + eps) / (np.sum(n*n) + eps))

def eval_pair(ref, est, sr=16000):
    m = min(len(ref), len(est))
    ref, est = ref[:m], est[:m]
    out = {}
    try:
        out['pesq_wb'] = float(pesq(sr, ref, est, 'wb'))
    except Exception:
        out['pesq_wb'] = np.nan
    try:
        out['stoi'] = float(stoi(ref, est, sr, extended=False))
    except Exception:
        out['stoi'] = np.nan
    try:
        out['si_sdr'] = float(si_sdr(ref, est))
    except Exception:
        out['si_sdr'] = np.nan
    return out

ckpt = torch.load(OUT_DIR / 'postfilter_best.pt', map_location=DEVICE)
postfilter.load_state_dict(ckpt['model'])
postfilter.eval()

rows_out = []
ex_dir = OUT_DIR / 'examples'
ex_dir.mkdir(parents=True, exist_ok=True)

with torch.no_grad():
    for i, batch in enumerate(tqdm(test_loader, desc='test eval')):
        if i >= MAX_EVAL_FILES:
            break
        clean = batch['clean'].to(DEVICE)
        base = dac_model(clean, sample_rate=SR)['audio']
        base, clean = align_len(base, clean)
        enh = postfilter(base)

        c = clean.squeeze().cpu().numpy()
        b = base.squeeze().cpu().numpy()
        e = enh.squeeze().cpu().numpy()

        mb = eval_pair(c, b, SR); mb['system'] = 'dac_baseline'
        me = eval_pair(c, e, SR); me['system'] = 'dac_postfilter'
        rows_out += [mb, me]

        if i < 5:
            sf.write(ex_dir / f'{i:03d}_clean.wav', c, SR)
            sf.write(ex_dir / f'{i:03d}_baseline.wav', b, SR)
            sf.write(ex_dir / f'{i:03d}_postfilter.wav', e, SR)

dfm = pd.DataFrame(rows_out)
summary = dfm.groupby('system').mean(numeric_only=True).reset_index()

# Define Google Drive output directory
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/nb_outputs_dac')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save to Google Drive
dfm.to_csv(DRIVE_OUTPUT_DIR / 'detailed_metrics.csv', index=False)
summary.to_csv(DRIVE_OUTPUT_DIR / 'metrics_summary.csv', index=False)

print(summary)
print('Saved detailed metrics to:', DRIVE_OUTPUT_DIR / 'detailed_metrics.csv')
print('Saved summary metrics to:', DRIVE_OUTPUT_DIR / 'metrics_summary.csv')


test eval:   8%|▊         | 40/502 [00:02<00:29, 15.71it/s]

           system   pesq_wb      stoi     si_sdr
0    dac_baseline  3.838658  0.950909 -12.148918
1  dac_postfilter  3.838659  0.950910 -12.148394
Saved detailed metrics to: /content/drive/MyDrive/nb_outputs_dac/detailed_metrics.csv
Saved summary metrics to: /content/drive/MyDrive/nb_outputs_dac/metrics_summary.csv


In [18]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [19]:
# Optional: IndicConformer WER/CER (requires model access and transcripts)
if RUN_ASR:
    from transformers import AutoModel
    from jiwer import wer, cer

    model_id = 'ai4bharat/indic-conformer-600m-multilingual'
    asr_model = AutoModel.from_pretrained(model_id, trust_remote_code=True)
    if hasattr(asr_model, 'to'):
        asr_model = asr_model.to(DEVICE)

    refs_base, hyps_base = [], []
    refs_pf, hyps_pf = [], []

    temp_dir = OUT_DIR / 'tmp_asr'
    temp_dir.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        for i, batch in enumerate(tqdm(test_loader, desc='asr eval')):
            if i >= min(MAX_EVAL_FILES, len(test)):
                break

            transcript = batch['transcript'][0].strip()
            if not transcript:
                continue

            clean = batch['clean'].to(DEVICE)
            base = dac_model(clean, sample_rate=SR)['audio']
            base, clean = align_len(base, clean)
            enh = postfilter(base)

            b = base.squeeze().cpu().numpy()
            e = enh.squeeze().cpu().numpy()
            bp = temp_dir / f'{i:04d}_base.wav'
            ep = temp_dir / f'{i:04d}_pf.wav'
            sf.write(bp, b, SR)
            sf.write(ep, e, SR)

            def transcribe(path):
                wav, _ = librosa.load(path, sr=16000, mono=True)
                w = torch.from_numpy(wav).unsqueeze(0).to(DEVICE)
                return str(asr_model(w, 'hi', 'ctc')).strip()

            pb = transcribe(str(bp))
            pp = transcribe(str(ep))

            refs_base.append(transcript); hyps_base.append(pb)
            refs_pf.append(transcript); hyps_pf.append(pp)

    if refs_base:
        asr_df = pd.DataFrame([
            {'system': 'dac_baseline', 'cer': cer(refs_base, hyps_base), 'wer': wer(refs_base, hyps_base)},
            {'system': 'dac_postfilter', 'cer': cer(refs_pf, hyps_pf), 'wer': wer(refs_pf, hyps_pf)},
        ])
        asr_df.to_csv(DRIVE_OUTPUT_DIR / 'asr_metrics.csv', index=False)
        print(asr_df)
    else:
        print('No usable transcripts found; ASR metrics skipped.')
else:
    print('RUN_ASR=False -> skipped ASR evaluation')

config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(

asr eval:   8%|▊         | 40/502 [00:16<03:09,  2.44it/s]

           system       cer       wer
0    dac_baseline  0.807523  0.828541
1  dac_postfilter  0.808219  0.828541


In [20]:
print('Done. Artifacts in:', OUT_DIR)
for p in sorted(OUT_DIR.glob('*')):
    print('-', p)

Done. Artifacts in: /content/nb_outputs
- /content/nb_outputs/examples
- /content/nb_outputs/postfilter_best.pt
- /content/nb_outputs/postfilter_last.pt
- /content/nb_outputs/tmp_asr


In [21]:
rows_noisy = []

def add_noise(wav, noise_level=0.01):
    noise = np.random.normal(0, noise_level, size=wav.shape).astype(np.float32)
    return np.clip(wav + noise, -1.0, 1.0)

with torch.no_grad():
    for i, batch in enumerate(tqdm(test_loader, desc='noisy eval')):
        if i >= MAX_EVAL_FILES:
            break

        clean = batch['clean'].to(DEVICE)

        c = clean.squeeze().cpu().numpy()

        c_noisy = add_noise(c, noise_level=0.01)

        clean_noisy = torch.from_numpy(c_noisy).unsqueeze(0).unsqueeze(0).to(DEVICE)

        base = dac_model(clean_noisy, sample_rate=SR)['audio']
        base, clean_noisy = align_len(base, clean_noisy)
        enh = postfilter(base)

        b = base.squeeze().cpu().numpy()
        e = enh.squeeze().cpu().numpy()

        mb = eval_pair(c_noisy, b, SR)
        mb['system'] = 'dac_baseline'
        mb['condition'] = 'noisy'

        me = eval_pair(c_noisy, e, SR)
        me['system'] = 'dac_postfilter'
        me['condition'] = 'noisy'

        rows_noisy += [mb, me]

dfm_noisy = pd.DataFrame(rows_noisy)
summary_noisy = dfm_noisy.groupby(['system', 'condition']).mean(numeric_only=True).reset_index()
dfm_noisy.to_csv(DRIVE_OUTPUT_DIR / 'detailed_metrics_noisy.csv', index=False)
summary_noisy.to_csv(DRIVE_OUTPUT_DIR / 'metrics_summary_noisy.csv', index=False)

print(summary_noisy)
print('Saved:', DRIVE_OUTPUT_DIR / 'metrics_summary_noisy.csv')

print(" Noisy eval done")

noisy eval:   8%|▊         | 40/502 [00:02<00:30, 15.39it/s]

           system condition   pesq_wb      stoi     si_sdr
0    dac_baseline     noisy  3.788413  0.904079 -12.465996
1  dac_postfilter     noisy  3.788431  0.904078 -12.465644
Saved: /content/drive/MyDrive/nb_outputs_dac/metrics_summary_noisy.csv
 Noisy eval done
